In [0]:
import csv, json, random
from datetime import datetime, timedelta

CATALOG = "livraison"          # ⚠️ "workspace" si tu n'as pas pu créer le catalog
RAW     = f"/Volumes/{CATALOG}/landing/raw"
FUTURE  = f"{RAW}/_future"
random.seed(42)                # même "hasard" à chaque exécution → résultats reproductibles

for sub in ["customers", "orders", "deliveries", "products",
            "_future/orders", "_future/deliveries"]:
    dbutils.fs.mkdirs(f"{RAW}/{sub}")

print("Dossiers créés dans", RAW)

In [0]:
FIRST  = ["Lina", "Karim", "Sofia", "Yanis", "Emma", "Nour", "Lucas", "Ines", "Adam", "Chloe"]
LAST   = ["Martin", "Benali", "Dubois", "Haddad", "Moreau", "Petit", "Traore", "Garcia"]
CITIES = {"Paris": "IDF", "Lyon": "ARA", "Marseille": "PACA", "Lille": "HDF", "Bordeaux": "NAQ"}

customers = []
for i in range(1, 501):
    fn, ln = random.choice(FIRST), random.choice(LAST)
    city   = random.choice(list(CITIES))
    d      = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 900))
    signup = d.strftime("%Y-%m-%d") if random.random() > 0.10 else d.strftime("%d/%m/%Y")  # 10 % de dates au format FR
    email  = f"{fn}.{ln}{i}@mail.com".lower()
    if random.random() < 0.05:
        email = "  " + email.upper() + " "                                   # emails mal formatés
    customers.append([f"C{i:04d}", fn, ln, email,
                      city + (" " if random.random() < 0.10 else ""),         # espaces parasites
                      CITIES[city], signup,
                      random.choice(["standard", "premium", None])])          # NULL

customers += random.sample(customers, 15)                                    # 15 doublons
customers += [[None, "Ghost", "User", "ghost@mail.com", "Paris", "IDF", "2025-01-01", "standard"]] * 3  # clé manquante

with open(f"{RAW}/customers/customers.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["customer_id", "first_name", "last_name", "email", "city", "region", "signup_date", "segment"])
    w.writerows(customers)

print(f"customers.csv : {len(customers)} lignes")

In [0]:
CATS = {"Pizza": (9, 16), "Burger": (8, 14), "Sushi": (12, 25), "Salade": (7, 12), "Dessert": (4, 8)}
products, pid = [], 1
for cat, (lo, hi) in CATS.items():
    for k in range(1, 5):
        products.append((f"P{pid:03d}", f"{cat} {k}", cat, round(random.uniform(lo, hi), 2)))
        pid += 1

(spark.createDataFrame(products, "product_id STRING, product_name STRING, category STRING, unit_price DOUBLE")
      .write.mode("overwrite").parquet(f"{RAW}/products/"))

print(f"products : {len(products)} produits écrits en Parquet")

In [0]:
price     = {p[0]: p[3] for p in products}   # product_id -> prix
order_num = 1
start     = datetime(2026, 9, 1)

for day in range(5):
    dt = start + timedelta(days=day)
    n  = 1000 if day == 0 else random.randint(100, 150)
    orders, deliveries = [], []

    for _ in range(n):
        oid = f"O{order_num:06d}"; order_num += 1
        pid_   = random.choice(list(price))
        qty    = random.randint(1, 4)
        amount = round(price[pid_] * qty, 2)
        ts     = dt + timedelta(minutes=random.randint(0, 1439))
        status = random.choice(["DELIVERED"] * 8 + ["CANCELLED", "PENDING"])

        r = random.random()
        if r < 0.03:   amount_str = str(-amount)                      # montant négatif
        elif r < 0.06: amount_str = str(amount).replace(".", ",")     # virgule décimale
        else:          amount_str = str(amount)

        ts_str = ts.strftime("%Y-%m-%d %H:%M:%S") if random.random() > 0.02 else "2026-13-45 99:99:99"  # date invalide
        cust   = f"C{random.randint(1, 520):04d}" if random.random() > 0.02 else None                 # C0501+ = client inconnu

        orders.append([oid, cust, pid_, qty, amount_str, ts_str, status])

        if status == "DELIVERED":
            d = {
                "delivery_id": f"D{oid[1:]}",
                "order_id": oid,
                "driver": {"id": f"DRV{random.randint(1, 30):02d}", "name": random.choice(FIRST)},
                "events": [
                    {"status": "PICKED_UP", "ts": (ts + timedelta(minutes=10)).isoformat()},
                    {"status": "DELIVERED", "ts": (ts + timedelta(minutes=random.randint(20, 70))).isoformat()},
                ],
                "rating": random.choice([1, 2, 3, 4, 5, None, "5"]),  # "5" = type incorrect
            }
            if random.random() < 0.05:
                d.pop("driver")                                          # champ manquant
            deliveries.append(d)

    orders += random.sample(orders, 10)                                  # doublons
    target = RAW if day == 0 else FUTURE

    with open(f"{target}/orders/orders_{dt:%Y-%m-%d}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["order_id", "customer_id", "product_id", "quantity", "amount", "order_ts", "status"])
        w.writerows(orders)

    with open(f"{target}/deliveries/deliveries_{dt:%Y-%m-%d}.json", "w") as f:
        for d in deliveries:
            f.write(json.dumps(d) + "\n")                                # JSON Lines : 1 objet par ligne

    print(f"{dt:%Y-%m-%d} → {len(orders)} commandes, {len(deliveries)} livraisons → {target}")

In [0]:
for sub in ["customers", "orders", "deliveries", "products", "_future/orders", "_future/deliveries"]:
    files = dbutils.fs.ls(f"{RAW}/{sub}")
    print(f"{sub:22s} → {len(files)} élément(s)")

print("\n--- Aperçu customers.csv ---")
print(open(f"{RAW}/customers/customers.csv").read()[:400])